### Alcance de este notebook
Aquí **sí** vamos a:
- cargar el dataset procesado del Sprint 2,
- cargar el preprocesador persistido,
- entrenar 5+ baselines con parámetros por defecto (solo estamos validando el flujo inicial, no buscando performance óptima todavía),
- guardar cada pipeline entrenado (que incluye feature engineering y preprocessing, porque esto asegura consistencia y evita data leakage en etapas posteriores) con `joblib`,
- validar mínimamente que cada modelo cargue y prediga,
- generar `models/baseline_manifest.csv` con meta-información para el siguiente rol,
- generar `models/baseline_smoke_test.csv` para comprobar la viabilidad.

Aquí **todavía no** vamos a:
- hacer tuning de hiperparámetros,
- hacer cross-validation completa,
- generar curvas ROC/PR o matrices de confusión,
- rankear modelos y seleccionar ganadores finales.

Estas exclusiones se hacen porque corresponden a roles posteriores (Metrics Evaluator).

## 1. Checklist antes de correr
Antes de ejecutar este notebook, valida:

1. Que estés dentro del entorno correcto del proyecto.
2. Que existen los archivos en `data/processed/`.
3. Que existe el preprocesador persistido.
4. Que conoces la columna objetivo (`TARGET_COL`).

> Si aún no sabes el target, **no sigas**: primero inspecciona `train_balanced.csv`.

In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

## 2. Configuración del proyecto
Ajusta estas rutas solo si tu repo usa una estructura distinta.

**Importante:** cambia `TARGET_COL` antes de entrenar.

In [2]:
PROJECT_ROOT = Path(".").resolve().parent if Path.cwd().name == "notebooks" else Path(".").resolve()
import sys
sys.path.append(str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

TRAIN_PATH = DATA_DIR / "train_balanced.csv"
TEST_PATH = DATA_DIR / "test_original.csv"

PREPROC_CANDIDATES = [
    MODELS_DIR / "preprocessor.pkl",
    MODELS_DIR / "preproc.pkl",
]

TARGET_COL = "IsBadBuy"

RANDOM_STATE = 42

In [3]:
print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_PATH exists:", TRAIN_PATH.exists(), TRAIN_PATH)
print("TEST_PATH exists:", TEST_PATH.exists(), TEST_PATH)
print("MODELS_DIR exists:", MODELS_DIR.exists(), MODELS_DIR)
print("Preprocessor candidates:")
for p in PREPROC_CANDIDATES:
    print(" -", p, "| exists:", p.exists())

PROJECT_ROOT: /Users/alexandralozano/dp261-g1
TRAIN_PATH exists: True /Users/alexandralozano/dp261-g1/data/processed/train_balanced.csv
TEST_PATH exists: True /Users/alexandralozano/dp261-g1/data/processed/test_original.csv
MODELS_DIR exists: True /Users/alexandralozano/dp261-g1/models
Preprocessor candidates:
 - /Users/alexandralozano/dp261-g1/models/preprocessor.pkl | exists: True
 - /Users/alexandralozano/dp261-g1/models/preproc.pkl | exists: False


## 3. Cargar insumos del Sprint 2
El Sprint 3 parte del dataset procesado y del pipeline/preprocesador generado en el Sprint 2.

In [4]:
def resolve_preprocessor_path(candidates):
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        "No se encontró el preprocesador. Revisar si el archivo se llama "
        "'preprocessor.pkl' o 'preproc.pkl'."
    )

PREPROCESSOR_PATH = resolve_preprocessor_path(PREPROC_CANDIDATES)
print("Usando preprocessor:", PREPROCESSOR_PATH)

Usando preprocessor: /Users/alexandralozano/dp261-g1/models/preprocessor.pkl


In [5]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH) if TEST_PATH.exists() else None

print("train_df shape:", train_df.shape)
if test_df is not None:
    print("test_df shape:", test_df.shape)

display(train_df.head())

train_df shape: (102410, 60)
test_df shape: (14597, 60)


,PurchDate,VehYear,VehicleAge,Make,Model,Trim,SubModel,WheelTypeID,VehOdo,Nationality,Size,TopThreeAmericanName,MMRAcquisitionAuctionAveragePrice,MMRAcquisitionAuctionCleanPrice,MMRAcquisitionRetailAveragePrice,MMRAcquisitonRetailCleanPrice,MMRCurrentAuctionAveragePrice,MMRCurrentAuctionCleanPrice,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost,edad_calculada,Transmission_MANUAL,Transmission_Manual,Color_BLACK,Color_BLUE,Color_BROWN,Color_GOLD,Color_GREEN,Color_GREY,Color_MAROON,Color_NOT AVAIL,Color_ORANGE,Color_OTHER,Color_PURPLE,Color_RED,Color_SILVER,Color_WHITE,Color_YELLOW,Auction_MANHEIM,Auction_OTHER,WheelType_Covers,WheelType_Special,odo_per_year,warranty_cost_ratio,cost_vs_acq_auction_avg_ratio,odometer_group,age_x_odo,cost_x_warranty,VehOdo_log1p,VehBCost_log1p,WarrantyCost_log1p,IsBadBuy
0,1970-01-01 00:00:01.261440000,2008.0,-38.0,0.128562,0.247487,0.134752,0.118644,1.0,85256.0,AMERICAN,1.0,CHRYSLER,9962.0,11419.0,14421.0,16012.0,9962.0,11419.0,14421.0,16012.0,NO,GREEN,0.093817,0.133803,UT,8880.0,1,2022.0,-38.0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,-2304.216216,0.227677,0.891298,alto,-3239728.0,17955360.0,11.353426,9.091669,7.612337,1
1,1970-01-01 00:00:01.265328000,2003.0,-33.0,0.117552,0.229729,0.122170,0.201183,1.0,68451.0,OTHER ASIAN,6.0,OTHER,4154.0,5253.0,7027.0,8313.0,4953.0,6253.0,7986.0,9583.0,NO,GREEN,0.139091,0.097002,NC,6300.0,0,1808.0,-33.0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,-2139.093750,0.286939,1.516245,medio_bajo,-2258883.0,11390400.0,11.133888,8.748464,7.500529,1
2,1970-01-01 00:00:01.240358400,2004.0,-34.0,0.154091,0.111732,0.147619,0.119658,2.0,94296.0,AMERICAN,11.0,FORD,3716.0,4933.0,4513.0,5828.0,3716.0,4933.0,4513.0,5828.0,NO,GREEN,0.119518,0.152381,AZ,4170.0,0,1774.0,-34.0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,True,False,-2857.454545,0.425318,1.121872,alto,-3206064.0,7397580.0,11.454205,8.335911,7.481556,0
3,1970-01-01 00:00:01.283990400,2003.0,-33.0,0.103237,0.097107,0.153110,0.112798,1.0,78498.0,AMERICAN,6.0,CHRYSLER,4293.0,5658.0,7647.0,8760.0,3930.0,5323.0,7242.0,8424.0,NO,GREEN,0.115777,0.104083,FL,6235.0,0,1689.0,-33.0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,-2453.062500,0.270847,1.452026,medio_alto,-2590434.0,10530915.0,11.270841,8.738094,7.432484,0
4,1970-01-01 00:00:01.273104000,2004.0,-34.0,0.103237,0.131547,0.115000,0.122195,1.0,80722.0,AMERICAN,8.0,CHRYSLER,6612.0,8094.0,9998.0,11289.0,4853.0,7462.0,8180.0,10573.0,NO,GREEN,0.168593,0.217016,CA,7565.0,0,1633.0,-34.0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,-2446.121212,0.215834,1.143959,medio_alto,-2744548.0,12353645.0,11.298779,8.931420,7.398786,1


## 4. Verificación del target y separación X / y

En este punto debes confirmar:
- cuál es la columna objetivo,
- que el target existe en `train_balanced.csv`,
- que `test_original.csv` no será usado para explorar métricas definitivas aquí.

In [6]:
TARGET_COL = "IsBadBuy"

if TARGET_COL not in train_df.columns:
    raise KeyError(f"El target '{TARGET_COL}' no existe en train_df.")

X_train = train_df.drop(columns=[TARGET_COL]).copy()
y_train = train_df[TARGET_COL].copy()

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("Target distribution:")
display(y_train.value_counts(dropna=False))

X_train shape: (102410, 59)
y_train shape: (102410,)
Target distribution:


IsBadBuy
1    51205
0    51205
Name: count, dtype: int64

## 5. Cargar el preprocesador persistido
El baseline debe usar un **pipeline completo** `preproc + clf`, no preprocesar por separado.

In [7]:
preprocessor = joblib.load(PREPROCESSOR_PATH)
print(type(preprocessor))
print(preprocessor)

<class 'dict'>
{'feature_builder': FeatureBuilder(), 'preprocessor': ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', RobustScaler())]),
                                 ['VehYear', 'VehicleAge', 'Make', 'Model',
                                  'Trim', 'SubModel', 'WheelTypeID', 'VehOdo',
                                  'Size', 'MMRAcquisitionAuctionAveragePrice',
                                  'MMRAcquisitionAuctionCleanPrice',
                                  'MMRAcquisitionRetailAveragePrice',
                                  'MMRAcquisitonRetailCleanPrice...
                                  'AUCGUART', 'VNST', 'Transmission_MANUAL',
                                  'Transmission_Manual', 'Color_BLACK',
                                  'Color_BLUE', 'Color_BROWN', 'Color_GOLD',

## 6. Registro de modelos baseline

Se incluyen:
- Logistic Regression
- Decision Tree
- Random Forest
- SVM
- KNN

Opcionales:
- Gaussian Naive Bayes
- Gradient Boosting

> Se usan parámetros por defecto o casi por defecto, añadiendo solo `random_state=42` cuando aplica para reproducibilidad.

> **Nota sobre SVC:** Se omitió `probability=True` debido a que el entrenamiento tomaba un tiempo excesivamente largo, bloqueando el progreso del notebook. Si en etapas posteriores se requiere `.predict_proba()` para curvas ROC/PR, se puede evaluar usar `CalibratedClassifierCV` o activar este parámetro asumiendo el costo computacional.

> Además, se agregó `max_iter=1000` al SVC para limitar el tiempo de entrenamiento, dado que el tamaño del dataset (`train_balanced.csv` con ~102k filas) hace que la optimización completa tarde un tiempo prohibitivo para un baseline.

In [8]:
model_registry = {
    "lr": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "dt": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "rf": RandomForestClassifier(random_state=RANDOM_STATE),
    "svm": SVC(max_iter=1000, random_state=RANDOM_STATE),
    "knn": KNeighborsClassifier(),
    # opcionales:
    # "nb": GaussianNB(),
    # "gb": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

list(model_registry.keys())

['lr', 'dt', 'rf', 'svm', 'knn']

## 7. Entrenamiento y persistencia de pipelines
Cada modelo se entrena como:
`Pipeline([("preproc", preprocessor), ("clf", estimator)])`

Esto asegura consistencia y evita fugas de información al pasar luego a evaluación.

In [9]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

trained_pipelines = {}
artifact_rows = []

for short_name, estimator in model_registry.items():
    # Defensive pipeline construction to handle both dictionary and single object preprocessors
    steps = []
    if isinstance(preprocessor, dict):
        if "feature_builder" in preprocessor:
            steps.append(("feature_builder", clone(preprocessor["feature_builder"])))
        if "preprocessor" in preprocessor:
            steps.append(("preproc", clone(preprocessor["preprocessor"])))
    else:
        steps.append(("preproc", clone(preprocessor)))
    
    steps.append(("clf", estimator))
    pipe = Pipeline(steps)

    print(f"Entrenando baseline_{short_name}.pkl ...")
    pipe.fit(X_train, y_train)

    output_path = MODELS_DIR / f"baseline_{short_name}.pkl"
    joblib.dump(pipe, output_path)

    trained_pipelines[short_name] = pipe
    artifact_rows.append(
        {
            "model_key": short_name,
            "estimator": estimator.__class__.__name__,
            "artifact_path": str(output_path),
            "target_col": TARGET_COL,
            "train_rows": X_train.shape[0],
            "train_columns": X_train.shape[1],
            "status": "trained",
            "notes": "Default baseline parameters"
        }
    )

artifacts_df = pd.DataFrame(artifact_rows)
artifacts_df.to_csv(MODELS_DIR / "baseline_manifest.csv", index=False)
display(artifacts_df)

Entrenando baseline_lr.pkl ...


Entrenando baseline_dt.pkl ...


Entrenando baseline_rf.pkl ...


Entrenando baseline_svm.pkl ...


Entrenando baseline_knn.pkl ...


,model_key,estimator,artifact_path,target_col,train_rows,train_columns,status,notes
0,lr,LogisticRegression,/Users/alexandralozano/dp261-g1/models/baselin...,IsBadBuy,102410,59,trained,Default baseline parameters
1,dt,DecisionTreeClassifier,/Users/alexandralozano/dp261-g1/models/baselin...,IsBadBuy,102410,59,trained,Default baseline parameters
2,rf,RandomForestClassifier,/Users/alexandralozano/dp261-g1/models/baselin...,IsBadBuy,102410,59,trained,Default baseline parameters
3,svm,SVC,/Users/alexandralozano/dp261-g1/models/baselin...,IsBadBuy,102410,59,trained,Default baseline parameters
4,knn,KNeighborsClassifier,/Users/alexandralozano/dp261-g1/models/baselin...,IsBadBuy,102410,59,trained,Default baseline parameters


## 8. Smoke test de carga y predicción
Esta validación mínima confirma que cada `.pkl`:
- se puede cargar,
- acepta datos crudos,
- genera predicciones.

La evaluación robusta viene en `10_evaluation.ipynb`.

In [10]:
sample_X = X_train.head(5).copy()

smoke_rows = []
for short_name in model_registry:
    artifact_path = MODELS_DIR / f"baseline_{short_name}.pkl"
    loaded_pipe = joblib.load(artifact_path)
    preds = loaded_pipe.predict(sample_X)

    smoke_rows.append(
        {
            "model_key": short_name,
            "artifact_exists": artifact_path.exists(),
            "n_predictions": len(preds),
            "sample_predictions": preds.tolist(),
        }
    )

smoke_df = pd.DataFrame(smoke_rows)
smoke_df.to_csv(MODELS_DIR / "baseline_smoke_test.csv", index=False)
display(smoke_df)

,model_key,artifact_exists,n_predictions,sample_predictions
0,lr,True,5,"[1, 1, 0, 1, 1]"
1,dt,True,5,"[1, 1, 0, 0, 1]"
2,rf,True,5,"[1, 1, 0, 0, 1]"
3,svm,True,5,"[1, 1, 1, 1, 1]"
4,knn,True,5,"[1, 1, 1, 0, 1]"


## 9. Resumen de salida del notebook
Este notebook debería dejar:
- pipelines baseline persistidos en `models/`,
- una tabla simple de artefactos generados,
- validación mínima de carga y predicción.

### Lo que sigue después
1. `10_evaluation.ipynb`
   - cross-validation estratificada,
   - métricas,
   - ROC / PR,
   - confusion matrix.

2. `11_model_comparison.ipynb`
   - tabla comparativa,
   - ranking,
   - top 2–3 candidatos.

3. `src/models.py` + `experiments_log.csv`
   - encapsular funciones,
   - log de experimentos,
   - versionado más ordenado para Sprint 4.

## 10. Notas para el PR
En tu Pull Request explica:
- qué archivo procesado usaste,
- cuál fue la columna target,
- qué nombre tenía finalmente el preprocesador (`preprocessor.pkl` o `preproc.pkl`),
- qué modelos entrenaste,
- qué artefactos se generaron en `models/`.